# Experiment 4 — Cross-Lingual Transfer: SQuAD English → Urdu UQA

## Why this experiment exists
There is no multilingual QA dataset containing Urdu (MLQA covers 7 languages but
not Urdu). The originally planned Experiment 3 (MLQA cross-dataset eval) was replaced
by this cross-lingual transfer experiment as the bonus contribution.

## The core idea
XLM-RoBERTa was pretrained on 100 languages including both English and Urdu.
Its weights already encode cross-lingual representations — English and Urdu tokens
are mapped into the same embedding space.

**Hypothesis:** Fine-tuning first on large English SQuAD (87k examples, high quality)
teaches the model *what span extraction looks like* before it ever sees Urdu.
When it then fine-tunes on Urdu UQA, it arrives with much stronger QA priors
and should converge faster and generalise better than direct UQA fine-tuning.

## Three runs — controlled comparison
| Run | Training path | Eval on |
|-----|--------------|---------|
| A (baseline) | XLM-R → UQA directly | UQA val |
| B (zero-shot transfer) | XLM-R → SQuAD only | UQA val |
| C (proposed method) | XLM-R → SQuAD → UQA | UQA val |

Run A = your existing A2/A3 result (64.45 EM / 77.14 F1) — no retraining needed.
Run B shows how much English SQuAD alone transfers to Urdu (zero-shot cross-lingual).
Run C is the proposed method — sequential transfer.

**If C > A:** English warm-starting helps Urdu QA. Cross-lingual transfer works.  
**If C ≈ A:** UQA is already sufficient. Transfer adds no benefit.  
**If B > 0 F1:** XLM-R's shared representations already encode cross-lingual QA signal.

Both outcomes are valid research findings.


In [1]:
!pip install transformers datasets accelerate evaluate sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


## 0. Mount Drive — all checkpoints persist here

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/cross-lingual-transfer'
SQUAD_DIR = f'{BASE}/run-B-squad-only'
TRANSFER_DIR = f'{BASE}/run-C-squad-then-uqa'
for d in [BASE, SQUAD_DIR, TRANSFER_DIR]:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

Mounted at /content/drive
Directories ready.


## 1. Config

In [3]:
MODEL   = "xlm-roberta-base"
MAX_LEN = 384
STRIDE  = 128
SEED    = 42

# Run B: SQuAD fine-tune settings
SQUAD_EPOCHS = 2      # 2 epochs on SQuAD — enough to learn span extraction
SQUAD_LR     = 2e-5

# Run C: UQA fine-tune settings (after SQuAD warm-start)
UQA_EPOCHS   = 4      # fewer epochs — model already knows QA from SQuAD
UQA_LR       = 1e-5   # lower LR — fine adjustment, not full learning

## 2. Load Tokenizer

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print("Tokenizer loaded:", MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: xlm-roberta-base


## 3. Shared Preprocessing Function (works for both SQuAD and UQA)

In [5]:
def preprocess(examples, tokenizer, max_length=MAX_LEN, stride=STRIDE):
    """
    Sliding-window tokenisation for extractive QA.
    Works identically for SQuAD (English) and UQA (Urdu) —
    both are SQuAD-format datasets with the same field names.
    """
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = inputs.pop("offset_mapping")
    sample_map     = inputs.pop("overflow_to_sample_mapping")
    answers        = examples["answers"]   # SQuAD uses 'answers' dict
    start_positions, end_positions = [], []

    for i, offset in enumerate(offset_mapping):
        sample_idx   = sample_map[i]
        answer       = answers[sample_idx]
        # SQuAD 'answers' = {'text': [...], 'answer_start': [...]}
        if len(answer["answer_start"]) == 0:
            start_positions.append(0); end_positions.append(0)
            continue
        start_char   = answer["answer_start"][0]
        end_char     = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1: idx += 1
        context_start = idx
        while sequence_ids[idx] == 1: idx += 1
        context_end = idx - 1

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0); end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char: idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char: idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"]   = end_positions
    return inputs

print("Preprocessing function defined.")

Preprocessing function defined.


## 4. Load SQuAD 2.0 (English) — for warm-start

SQuAD 2.0 has 87,599 answerable + 53,775 unanswerable questions.
We filter to answerable-only (same protocol as UQA experiments)
so the task format is identical for the transfer.


In [6]:
from datasets import load_dataset

print("Loading SQuAD 2.0...")
squad_raw = load_dataset("rajpurkar/squad_v2")

# Filter to answerable only — same protocol as UQA
squad_raw["train"] = squad_raw["train"].filter(
    lambda x: len(x["answers"]["answer_start"]) > 0
)
squad_raw["validation"] = squad_raw["validation"].filter(
    lambda x: len(x["answers"]["answer_start"]) > 0
)
print(squad_raw)
print(f"\nSQuAD answerable train   : {len(squad_raw['train']):,}")
print(f"SQuAD answerable val     : {len(squad_raw['validation']):,}")

Loading SQuAD 2.0...


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11873 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 86821
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5928
    })
})

SQuAD answerable train   : 86,821
SQuAD answerable val     : 5,928


In [7]:
from functools import partial

preprocess_fn = partial(preprocess, tokenizer=tokenizer)

squad_train = squad_raw["train"].map(
    preprocess_fn, batched=True,
    remove_columns=squad_raw["train"].column_names
)
squad_val = squad_raw["validation"].map(
    preprocess_fn, batched=True,
    remove_columns=squad_raw["validation"].column_names
)
print(f"SQuAD train windows : {len(squad_train):,}")
print(f"SQuAD val windows   : {len(squad_val):,}")

Map:   0%|          | 0/86821 [00:00<?, ? examples/s]

Map:   0%|          | 0/5928 [00:00<?, ? examples/s]

SQuAD train windows : 88,765
SQuAD val windows   : 6,178


## 5. Load UQA (Urdu) — for second stage of transfer and final eval

In [8]:
def uqa_to_squad_format(example):
    """Convert UQA flat fields to SQuAD answers-dict format for the shared preprocessor."""
    return {
        "id":       example["id"],
        "title":    example["title"],
        "context":  example["context"],
        "question": example["question"],
        "answers": {
            "text":         [example["answer"]],
            "answer_start": [example["answer_start"]],
        }
    }

print("Loading UQA...")
uqa_raw = load_dataset("uqa/UQA")
uqa_raw["train"]      = uqa_raw["train"].filter(lambda x: not x["is_impossible"])
uqa_raw["validation"] = uqa_raw["validation"].filter(lambda x: not x["is_impossible"])

# Convert to SQuAD format so the same preprocessor works
uqa_train_sq = uqa_raw["train"].map(uqa_to_squad_format,
                                     remove_columns=uqa_raw["train"].column_names)
uqa_val_sq   = uqa_raw["validation"].map(uqa_to_squad_format,
                                          remove_columns=uqa_raw["validation"].column_names)

print(f"UQA train : {len(uqa_train_sq):,}")
print(f"UQA val   : {len(uqa_val_sq):,}")

Loading UQA...


README.md:   0%|          | 0.00/898 [00:00<?, ?B/s]

data/train-00000-of-00001-bac007e8ca7192(…):   0%|          | 0.00/30.2M [00:00<?, ?B/s]

data/validation-00000-of-00001-cf8a6960d(…):   0%|          | 0.00/2.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/124745 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/124745 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16824 [00:00<?, ? examples/s]

Map:   0%|          | 0/83018 [00:00<?, ? examples/s]

Map:   0%|          | 0/11169 [00:00<?, ? examples/s]

UQA train : 83,018
UQA val   : 11,169


In [9]:
uqa_train_tok = uqa_train_sq.map(
    preprocess_fn, batched=True,
    remove_columns=uqa_train_sq.column_names
)
uqa_val_tok = uqa_val_sq.map(
    preprocess_fn, batched=True,
    remove_columns=uqa_val_sq.column_names
)
print(f"UQA train windows : {len(uqa_train_tok):,}")
print(f"UQA val windows   : {len(uqa_val_tok):,}")

Map:   0%|          | 0/83018 [00:00<?, ? examples/s]

Map:   0%|          | 0/11169 [00:00<?, ? examples/s]

UQA train windows : 86,930
UQA val windows   : 11,884


## 6. Evaluation Helper

In [10]:
import torch, evaluate as hf_evaluate
from datasets import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

squad_metric = hf_evaluate.load("squad")

def build_uqa_eval_dataset(raw_val):
    """Merge duplicate question IDs so each question has all answer variants."""
    data = raw_val.to_dict()
    grouped = {}
    for i in range(len(data['id'])):
        idx = data['id'][i]
        if idx not in grouped:
            grouped[idx] = {
                'id': idx, 'context': data['context'][i],
                'question': data['question'][i],
                'answer': [data['answers'][i]['text'][0]],
                'answer_start': [data['answers'][i]['answer_start'][0]],
            }
        else:
            grouped[idx]['answer'].append(data['answers'][i]['text'][0])
            grouped[idx]['answer_start'].append(data['answers'][i]['answer_start'][0])
    return Dataset.from_dict(
        {k: [d[k] for d in grouped.values()] for k in list(grouped.values())[0]}
    )

def run_eval(model_path, eval_dataset, label="", batch_size=16):
    """Evaluate a saved model checkpoint on any SQuAD-format eval dataset."""
    print(f"\n{'='*58}")
    print(f"  {label}")
    print(f"  Questions : {len(eval_dataset):,}")
    print(f"{'='*58}")
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForQuestionAnswering.from_pretrained(model_path).to("cuda")
    mdl.eval()
    predictions, references = [], []
    for start in tqdm(range(0, len(eval_dataset), batch_size)):
        batch = eval_dataset[start: start + batch_size]
        enc = tok(
            batch["question"], batch["context"],
            padding="max_length", truncation=True,
            max_length=MAX_LEN, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            out = mdl(**enc)
        for i in range(len(batch["id"])):
            s = torch.argmax(out.start_logits[i]).item()
            e = torch.argmax(out.end_logits[i]).item()
            if e < s: e = s
            pred = tok.decode(enc["input_ids"][i][s:e+1], skip_special_tokens=True)
            predictions.append({"id": batch["id"][i], "prediction_text": pred})
            references.append({
                "id": batch["id"][i],
                "answers": {"text": batch["answer"][i], "answer_start": batch["answer_start"][i]}
            })
    results = squad_metric.compute(predictions=predictions, references=references)
    print(f"  EM = {results['exact_match']:.4f}   F1 = {results['f1']:.4f}")
    return results

# Build the UQA eval dataset once — reused for all 3 runs
uqa_eval_ds = build_uqa_eval_dataset(uqa_val_sq)
print(f"\nUQA eval dataset: {len(uqa_eval_ds):,} unique questions")


UQA eval dataset: 5,811 unique questions


## 7. Trainer Builder Helper

In [11]:
from transformers import (TrainingArguments, Trainer,
                        AutoModelForQuestionAnswering,
                        default_data_collator, EarlyStoppingCallback)
from transformers.trainer_utils import get_last_checkpoint

def build_trainer(model_path, output_dir, train_ds, eval_ds,
                  epochs, lr, patience=4):
    """
    Builds a Trainer.
    model_path: HF hub name OR local path to existing checkpoint to continue from.
    """
    model = AutoModelForQuestionAnswering.from_pretrained(model_path)
    args = TrainingArguments(
        output_dir=output_dir,
        seed=SEED,
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=0.01,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=8,
        fp16=True,
        eval_strategy="steps", eval_steps=200,
        save_strategy="steps", save_steps=200,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=100,
        report_to="none",
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=default_data_collator,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)]
    )
    return trainer, model

## 8. Run B — SQuAD-only fine-tune → evaluate on UQA (zero-shot cross-lingual)

This answers: *Does English QA training alone transfer to Urdu at all?*  
No Urdu data used in training. Pure cross-lingual zero-shot.  
Expected: F1 somewhere between 30–55 depending on XLM-R's Urdu coverage.


In [12]:
# ── Stage 1: Fine-tune on SQuAD (English) ───────────────────────────────────
print("RUN B — Stage 1: Fine-tuning on SQuAD 2.0 (English)...")
print(f"Training samples : {len(squad_train):,} windows")
print(f"Epochs           : {SQUAD_EPOCHS}")
print(f"Learning rate    : {SQUAD_LR}")
print()

trainer_B, model_B = build_trainer(
    model_path  = MODEL,         # start from pretrained XLM-R
    output_dir  = SQUAD_DIR,
    train_ds    = squad_train,
    eval_ds     = squad_val,
    epochs      = SQUAD_EPOCHS,
    lr          = SQUAD_LR,
)

last_ckpt = get_last_checkpoint(SQUAD_DIR)
print(f"Resuming from: {last_ckpt}" if last_ckpt else "Starting fresh")
trainer_B.train(resume_from_checkpoint=last_ckpt)

# Save best weights
SQUAD_BEST = f'{BASE}/run-B-best'
os.makedirs(SQUAD_BEST, exist_ok=True)
trainer_B.model.save_pretrained(SQUAD_BEST)
tokenizer.save_pretrained(SQUAD_BEST)
print(f"\nRun B model saved to: {SQUAD_BEST}")

RUN B — Stage 1: Fine-tuning on SQuAD 2.0 (English)...
Training samples : 88,765 windows
Epochs           : 2
Learning rate    : 2e-05



model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
qa_outputs.weight           | MISSING    | 
qa_outputs.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fresh


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Run B model saved to: /content/drive/MyDrive/cross-lingual-transfer/run-B-best


In [13]:
# ── Evaluate Run B on UQA (zero-shot — no Urdu training) ────────────────────
print("RUN B — Evaluating SQuAD-trained model on UQA Urdu (zero-shot)...")
results_B = run_eval(SQUAD_BEST, uqa_eval_ds,
                     label="Run B: SQuAD-only → UQA zero-shot")

RUN B — Evaluating SQuAD-trained model on UQA Urdu (zero-shot)...

  Run B: SQuAD-only → UQA zero-shot
  Questions : 5,811


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 364/364 [02:31<00:00,  2.39it/s]


  EM = 49.0621   F1 = 63.6081


## 9. Run C — SQuAD warm-start → then UQA (proposed method)

Start from the SQuAD-trained model (Run B) and continue fine-tuning on Urdu UQA.  
Lower LR (1e-5 vs 2e-5) — we want fine adjustment, not overwriting the SQuAD knowledge.  
Fewer epochs (4 vs 6) — the model already knows QA from SQuAD, converges faster.


In [14]:
# ── Stage 2: Continue fine-tuning on UQA (Urdu) from SQuAD checkpoint ──────
print("RUN C — Stage 2: Fine-tuning on UQA (Urdu) from SQuAD warm-start...")
print(f"Starting from    : {SQUAD_BEST}  (SQuAD-trained)")
print(f"Training samples : {len(uqa_train_tok):,} windows")
print(f"Epochs           : {UQA_EPOCHS}")
print(f"Learning rate    : {UQA_LR}  (lower than Stage 1 — fine adjustment)")
print()

trainer_C, model_C = build_trainer(
    model_path  = SQUAD_BEST,   # ← KEY: start from SQuAD checkpoint, not pretrained
    output_dir  = TRANSFER_DIR,
    train_ds    = uqa_train_tok,
    eval_ds     = uqa_val_tok,
    epochs      = UQA_EPOCHS,
    lr          = UQA_LR,
)

last_ckpt = get_last_checkpoint(TRANSFER_DIR)
print(f"Resuming from: {last_ckpt}" if last_ckpt else "Starting fresh")
trainer_C.train(resume_from_checkpoint=last_ckpt)

# Save best weights
TRANSFER_BEST = f'{BASE}/run-C-best'
os.makedirs(TRANSFER_BEST, exist_ok=True)
trainer_C.model.save_pretrained(TRANSFER_BEST)
tokenizer.save_pretrained(TRANSFER_BEST)
print(f"\nRun C model saved to: {TRANSFER_BEST}")

RUN C — Stage 2: Fine-tuning on UQA (Urdu) from SQuAD warm-start...
Starting from    : /content/drive/MyDrive/cross-lingual-transfer/run-B-best  (SQuAD-trained)
Training samples : 86,930 windows
Epochs           : 4
Learning rate    : 1e-05  (lower than Stage 1 — fine adjustment)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Starting fresh


Step,Training Loss,Validation Loss
200,1.433890,1.687306
400,1.339814,1.621128
600,1.304410,1.597164
800,1.214012,1.605843
1000,1.180990,1.573497
1200,1.156226,1.556962


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Run C model saved to: /content/drive/MyDrive/cross-lingual-transfer/run-C-best


In [19]:
# ── Evaluate Run C on UQA ────────────────────────────────────────────────────
print("RUN C — Evaluating SQuAD→UQA transfer model on UQA...")
results_C = run_eval("/content/drive/MyDrive/cross-lingual-transfer/run-C-squad-then-uqa/checkpoint-1600", uqa_eval_ds,
                     label="Run C: SQuAD→UQA transfer (proposed method)")

RUN C — Evaluating SQuAD→UQA transfer model on UQA...

  Run C: SQuAD→UQA transfer (proposed method)
  Questions : 5,811


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 364/364 [02:32<00:00,  2.39it/s]


  EM = 62.8463   F1 = 76.1228


## 10. Final Comparison Table

In [20]:
# Run A baseline — from your existing A2/A3 experiments
RUN_A_EM = 64.45   # from A2 XLM-R reproduction
RUN_A_F1 = 77.14   # update this if you re-evaluated with the fixed eval in Exp 2

print()
print("=" * 62)
print("  CROSS-LINGUAL TRANSFER — FINAL RESULTS")
print("=" * 62)
print(f"  {'Run':<42} {'EM':>8} {'F1':>8}")
print("-" * 62)
print(f"  {'Run A: XLM-R → UQA direct (A2 baseline)':<42} {RUN_A_EM:>8.2f} {RUN_A_F1:>8.2f}")
print(f"  {'Run B: XLM-R → SQuAD only (zero-shot Urdu)':<42} {results_B['exact_match']:>8.2f} {results_B['f1']:>8.2f}")
print(f"  {'Run C: XLM-R → SQuAD → UQA (proposed)':<42} {results_C['exact_match']:>8.2f} {results_C['f1']:>8.2f}")
print("-" * 62)

# Analysis
transfer_gain = results_C['f1'] - RUN_A_F1
zeroshot_f1   = results_B['f1']

print(f"\n  Transfer gain (C vs A) : {transfer_gain:+.2f} F1")
print(f"  Zero-shot cross-lingual: {zeroshot_f1:.2f} F1  (Run B, no Urdu training)")
print()

if transfer_gain > 1.5:
    finding = "POSITIVE TRANSFER: English warm-start improves Urdu QA."
    explanation = ("XLM-R's shared multilingual representations allow span-extraction "
                   "patterns learned in English to transfer to Urdu, reducing the "
                   "effective amount of Urdu data needed for convergence.")
elif transfer_gain > -1.5:
    finding = "NEUTRAL TRANSFER: No significant improvement or degradation."
    explanation = ("UQA (83k examples) is large enough that the model converges "
                   "to the same solution regardless of English warm-starting. "
                   "The English SQuAD signal is subsumed by the Urdu training.")
else:
    finding = "NEGATIVE TRANSFER: English warm-start slightly hurts Urdu QA."
    explanation = ("English SVO syntax may bias the span-prediction head in ways "
                   "that interfere with Urdu SOV patterns. The lower LR in Stage 2 "
                   "may not have been sufficient to override this bias.")

print(f"  Finding    : {finding}")
print(f"  Explanation: {explanation}")
print("=" * 62)


  CROSS-LINGUAL TRANSFER — FINAL RESULTS
  Run                                              EM       F1
--------------------------------------------------------------
  Run A: XLM-R → UQA direct (A2 baseline)       64.45    77.14
  Run B: XLM-R → SQuAD only (zero-shot Urdu)    49.06    63.61
  Run C: XLM-R → SQuAD → UQA (proposed)         62.85    76.12
--------------------------------------------------------------

  Transfer gain (C vs A) : -1.02 F1
  Zero-shot cross-lingual: 63.61 F1  (Run B, no Urdu training)

  Finding    : NEUTRAL TRANSFER: No significant improvement or degradation.
  Explanation: UQA (83k examples) is large enough that the model converges to the same solution regardless of English warm-starting. The English SQuAD signal is subsumed by the Urdu training.
